# Cellpose-SAM 920-nm green sessions — WSL v2

WSL notebook for local CellposeSAM segmentation on the RTX 3050.

Input:
`/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/920/preprocessing/green.tif`

Output:
`/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/920/segmentation/mask.tif`

Behavior preserved from the established workflow:
- `cpsam_v2`
- `do_3D=True`
- `z_axis=0`
- `channel_axis=3`
- `min_size=100`
- existing `mask.tif` files are skipped unless `OVERWRITE_EXISTING=True`

This v2 also:
- logs runtime per newly segmented session
- logs peak CUDA allocated/reserved memory
- releases large per-session objects after every stack
- runs `gc.collect()` and `torch.cuda.empty_cache()` between stacks


In [ ]:
from pathlib import Path
import csv
import gc
import os
import time
import uuid

import numpy as np
import tifffile
import torch
from tqdm.auto import tqdm
from cellpose import models
from cellpose.io import imread_3D


In [ ]:
DERIVATIVES_ROOT = Path("/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives")

# None = discover all mice with matching data.
MOUSE_IDS = None

OVERWRITE_EXISTING = False
MIN_SIZE = 100

if not DERIVATIVES_ROOT.is_dir():
    raise FileNotFoundError(f"Derivatives root not found: {DERIVATIVES_ROOT}")


In [ ]:
# Discover canonical 920-nm green stacks.
image_paths = []

mouse_dirs = sorted(
    p for p in DERIVATIVES_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith("_")
)

if MOUSE_IDS is not None:
    wanted = set(MOUSE_IDS)
    mouse_dirs = [p for p in mouse_dirs if p.name in wanted]

for mouse_dir in mouse_dirs:
    sessions_dir = mouse_dir / "sessions"
    if not sessions_dir.is_dir():
        continue

    for session_dir in sorted(p for p in sessions_dir.iterdir() if p.is_dir()):
        image_path = session_dir / "920" / "preprocessing" / "green.tif"
        if image_path.is_file():
            image_paths.append(image_path)

if not image_paths:
    raise FileNotFoundError("No matching 920-nm green stacks were found.")

print(f"Found {len(image_paths)} stack(s).")
for p in image_paths:
    print(p)


In [ ]:
model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)

print("Model device:", model.device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
results = []

for image_path in tqdm(image_paths, desc="Cellpose-SAM 920 sessions"):
    mouse_id = image_path.parents[4].name
    session_id = image_path.parents[2].name
    segmentation_dir = image_path.parent.parent / "segmentation"
    output_path = segmentation_dir / "mask.tif"

    if output_path.exists() and not OVERWRITE_EXISTING:
        tqdm.write(f"Skipping existing: {mouse_id} {session_id} -> {output_path}")
        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(image_path),
            "mask_path": str(output_path),
            "status": "ALREADY_EXISTS",
            "n_masks": "",
            "runtime_min": "",
            "peak_allocated_gb": "",
            "peak_reserved_gb": "",
            "error": "",
        })
        continue

    loaded_image = masks = flows = styles = mask_to_save = check = None
    temp_path = None

    try:
        tqdm.write(f"Processing: {mouse_id} {session_id}")
        loaded_image = imread_3D(image_path.as_posix())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        masks, flows, styles = model.eval(
            loaded_image,
            do_3D=True,
            z_axis=0,
            channel_axis=3,
            min_size=MIN_SIZE,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        runtime_min = (time.perf_counter() - t0) / 60.0
        n_masks = int(np.max(masks))

        if torch.cuda.is_available():
            peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
            peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
        else:
            peak_allocated_gb = float("nan")
            peak_reserved_gb = float("nan")

        tqdm.write(
            f"  Found {n_masks} mask(s) | Runtime: {runtime_min:.2f} min | "
            f"Peak GPU: {peak_allocated_gb:.2f} GB allocated, "
            f"{peak_reserved_gb:.2f} GB reserved"
        )

        segmentation_dir.mkdir(parents=True, exist_ok=True)

        dtype = np.uint16 if n_masks <= np.iinfo(np.uint16).max else np.uint32
        mask_to_save = masks.astype(dtype, copy=False)

        temp_path = segmentation_dir / f".mask.tif.tmp.{uuid.uuid4().hex}"

        tifffile.imwrite(
            temp_path,
            mask_to_save,
            photometric="minisblack",
            metadata={"axes": "ZYX"},
        )

        check = tifffile.imread(temp_path)

        if check.shape != mask_to_save.shape:
            raise RuntimeError(
                f"Saved mask shape mismatch: expected {mask_to_save.shape}, got {check.shape}"
            )

        if int(np.max(check)) != n_masks:
            raise RuntimeError(
                f"Saved mask label check failed: expected {n_masks}, got {int(np.max(check))}"
            )

        if output_path.exists():
            raise FileExistsError(
                f"Destination appeared during processing; refusing to overwrite: {output_path}"
            )

        os.rename(temp_path, output_path)
        temp_path = None

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(image_path),
            "mask_path": str(output_path),
            "status": "SEGMENTED",
            "n_masks": n_masks,
            "runtime_min": round(runtime_min, 3),
            "peak_allocated_gb": round(peak_allocated_gb, 3),
            "peak_reserved_gb": round(peak_reserved_gb, 3),
            "error": "",
        })

        tqdm.write(f"Saved: {output_path}")

    except Exception as error:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "green_path": str(image_path),
            "mask_path": str(output_path),
            "status": "FAILED",
            "n_masks": "",
            "runtime_min": "",
            "peak_allocated_gb": "",
            "peak_reserved_gb": "",
            "error": str(error),
        })

        tqdm.write(f"FAILED: {mouse_id} {session_id}")
        tqdm.write(f"  {error}")

    finally:
        # Release large per-session objects but keep the Cellpose model loaded.
        loaded_image = None
        masks = None
        flows = None
        styles = None
        mask_to_save = None
        check = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            allocated_gb = torch.cuda.memory_allocated() / 1024**3
            reserved_gb = torch.cuda.memory_reserved() / 1024**3
            tqdm.write(
                f"  GPU after cleanup: {allocated_gb:.2f} GB allocated, "
                f"{reserved_gb:.2f} GB reserved"
            )


In [ ]:
log_path = DERIVATIVES_ROOT / "cellposeSAM_920_session_log.csv"

fieldnames = [
    "mouse_id",
    "session_id",
    "green_path",
    "mask_path",
    "status",
    "n_masks",
    "runtime_min",
    "peak_allocated_gb",
    "peak_reserved_gb",
    "error",
]

with log_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print("\nFinished.")
print("Log:", log_path)

status_counts = {}
for row in results:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1

for status, count in sorted(status_counts.items()):
    print(f"{status:20s} {count:4d}")
